# 🍕 Food Bank of the World 🌍 — Limpieza de Datos

## Descripción general

Es un conjunto de datos disponible en Kaggle que recopila información sobre la producción mundial de alimentos durante más de 60 años **(1961–2020)**. Incluye más de 2 millones de registros relacionados con cultivos, ganadería, uso de la tierra y población en 247+ países y territorios.

## Objetivo

El dataset busca ayudar a investigadores, analistas y científicos de datos a estudiar:
1. Producción agrícola mundial.
2. Seguridad alimentaria.
3. Uso eficiente de recursos naturales.
4. Impacto del crecimiento poblacional sobre la producción de alimentos.
5. Tendencias históricas de cultivos y ganadería.
6. Modelos predictivos de oferta alimentaria.

La inspiración surge de la preocupación por la crisis alimentaria mundial, agravada por factores como:
1. Conflictos internacionales (ej. guerra en Ucrania).
2. Problemas en cadenas de suministro.
3. Consecuencias económicas de la pandemia de COVID-19.
4. Incremento global de los precios de los alimentos.

---
# 🌾 PARTE 1 — crop1.csv
### Producción Agrícola Mundial

Contiene datos históricos de producción agrícola de más de 150 cultivos en más de 247 países, cubriendo más de seis décadas (1961–2020). Incluye indicadores como producción, rendimiento y área cosechada.

### Carga de datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ruta del archivo
ruta_crop = r"C:\Users\ESTUDIANTE\Downloads\dataset cliente1\crop1.csv"

# Cargar CSV
df_crop = pd.read_csv(ruta_crop)

# Mostrar las primeras 20 filas para identificar datos sucios
df_crop.head(20)

### Revisión de estructura y valores nulos

- `df.info()` → Muestra la estructura del dataset, tipos de datos, cantidad de registros y valores no nulos por columna.
- `df.isnull().sum()` → Cuenta la cantidad de valores nulos (faltantes) existentes en cada columna del dataset.

In [ ]:
# Revisar estructura y tipos de datos
df_crop.info()

In [ ]:
# Contar valores nulos por columna
print(df_crop.isnull().sum())

### Eliminamos filas con valores nulos en la columna Value

In [ ]:
# Eliminar filas con valores nulos en la columna Value
df_crop = df_crop.dropna(subset=['Value'])

In [ ]:
# Corroborar que se eliminaron
print(df_crop.isnull().sum())

### Revisamos y eliminamos registros duplicados

In [ ]:
# Contar duplicados antes de eliminar
print("Duplicados encontrados:", df_crop.duplicated().sum())

In [ ]:
# Eliminar duplicados
df_crop = df_crop.drop_duplicates()
print("Duplicados eliminados. Registros restantes:", len(df_crop))

### Creamos una copia de trabajo

In [ ]:
# Copia del DataFrame limpio
df_crop1 = df_crop.copy()
df_crop1

In [ ]:
df_crop1.info()

In [ ]:
df_crop1.describe()

### Corregimos tipos de datos y limpiamos espacios en columnas de texto

In [ ]:
# Eliminar espacios extra en columnas de texto
df_crop1['Area']    = df_crop1['Area'].str.strip()
df_crop1['Item']    = df_crop1['Item'].str.strip()
df_crop1['Element'] = df_crop1['Element'].str.strip()

# Convertir Year a datetime
df_crop1['Year'] = pd.to_datetime(df_crop1['Year'], format='%Y')

# Asegurar que Value sea numérico
df_crop1['Value'] = pd.to_numeric(df_crop1['Value'], errors='coerce')

df_crop1.head()

### Obtenemos los valores únicos de la columna Element

In [ ]:
# Ver los indicadores únicos del dataset
valores_unicos_element = df_crop1['Element'].unique()
print(valores_unicos_element)

### Validamos rangos lógicos — eliminamos valores negativos

In [ ]:
# Revisar rango de años
df_crop1['Year_int'] = df_crop1['Year'].dt.year
print("Rango de años:", df_crop1['Year_int'].min(), "→", df_crop1['Year_int'].max())

# Revisar valores negativos (no tienen sentido para producción/rendimiento)
print("Valores negativos en Value:", (df_crop1['Value'] < 0).sum())

# Eliminar negativos si existen
df_crop1 = df_crop1[df_crop1['Value'] >= 0]
print("Registros tras eliminar negativos:", len(df_crop1))

### Eliminamos outliers con umbral en el percentil 99

In [ ]:
# Revisar estadísticas antes del filtro
df_crop1.describe()

In [ ]:
# Aplicar umbral en percentil 99 para eliminar outliers extremos
umbral_crop = df_crop1['Value'].quantile(0.99)
df_crop1 = df_crop1[df_crop1['Value'] < umbral_crop]

print("Umbral aplicado:", umbral_crop)
df_crop1.describe()

### Renombramos columnas al español

In [ ]:
# Mapeo de nombres al español
mapeo_element_crop = {
    'Area harvested': 'Área Cosechada',
    'Production':     'Producción',
    'Yield':          'Rendimiento'
}

df_crop1['Element'] = df_crop1['Element'].map(mapeo_element_crop).fillna(df_crop1['Element'])

# Renombrar columnas base
df_crop1 = df_crop1.rename(columns={
    'Area':    'País',
    'Item':    'Cultivo',
    'Element': 'Indicador',
    'Unit':    'Unidad',
    'Value':   'Valor'
})

# Eliminar columna auxiliar Year_int
df_crop1 = df_crop1.drop(columns=['Year_int'])

df_crop1.head()

### Separamos en dos DataFrames

In [ ]:
# DataFrame 1: columnas base de identificación
df_crop1a = df_crop1[['País', 'Cultivo', 'Indicador', 'Year', 'Unidad', 'Valor']].copy()

# DataFrame 2: tabla pivote por indicador
df_crop1b = df_crop1.pivot_table(
    index=['País', 'Cultivo', 'Year'],
    columns='Indicador',
    values='Valor',
    aggfunc='mean'
).reset_index()

print("df_crop1a shape:", df_crop1a.shape)
print("df_crop1b shape:", df_crop1b.shape)
df_crop1b.head()

---
# 📊 Visualización Global — crop1.csv

In [ ]:
# Valor medio por año
df_crop1a['Year_int'] = df_crop1a['Year'].dt.year
prod_por_año = df_crop1a.groupby('Year_int')['Valor'].mean().reset_index()

plt.figure(figsize=(15, 6))
sns.lineplot(data=prod_por_año, x='Year_int', y='Valor')
plt.title('Valor Medio por Año — Producción Agrícola (1961–2020)')
plt.xlabel('Año')
plt.ylabel('Valor Medio')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 países por valor medio
top_paises_crop = df_crop1a.groupby('País')['Valor'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_paises_crop.values, y=top_paises_crop.index)
plt.title('Top 10 Países por Valor Medio — Cultivos')
plt.xlabel('Valor Medio')
plt.ylabel('País')
plt.tight_layout()
plt.show()

In [ ]:
# Valor medio por indicador
por_indicador_crop = df_crop1a.groupby('Indicador')['Valor'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=por_indicador_crop.values, y=por_indicador_crop.index)
plt.title('Valor Medio por Indicador — Cultivos')
plt.xlabel('Valor Medio')
plt.ylabel('Indicador')
plt.tight_layout()
plt.show()

---
# 🐄 PARTE 2 — live1.csv
### Producción Ganadera Mundial

Contiene datos históricos de existencias y producción ganadera en más de 247 países durante más de seis décadas. Incluye información sobre diferentes tipos de animales de granja y sus indicadores productivos.

### Carga de datos

In [ ]:
# Ruta del archivo
ruta_live = r"C:\Users\ESTUDIANTE\Downloads\dataset cliente1\live1.csv"

# Cargar CSV
df_live = pd.read_csv(ruta_live)

# Mostrar las primeras 20 filas para identificar datos sucios
df_live.head(20)

### Revisión de estructura y valores nulos

- `df.info()` → Muestra la estructura del dataset, tipos de datos, cantidad de registros y valores no nulos por columna.
- `df.isnull().sum()` → Cuenta la cantidad de valores nulos (faltantes) existentes en cada columna del dataset.

In [ ]:
# Revisar estructura y tipos de datos
df_live.info()

Vemos que hay 122.458 registros totales con 120.194 valores no nulos, lo cual nos indica que hay **2.264 valores nulos** en la columna Value.

Corroboramos los datos faltantes con `df.isnull().sum()`

In [ ]:
df_live.isnull().sum()

### Eliminamos filas con valores nulos en Value

In [ ]:
df_live = df_live.dropna(subset=['Value'])

In [ ]:
# Corroborar que se eliminaron
df_live.isnull().sum()

### Revisamos y eliminamos registros duplicados

In [ ]:
# Contar duplicados
print("Duplicados encontrados:", df_live.duplicated().sum())

In [ ]:
# Eliminar duplicados
df_live = df_live.drop_duplicates()
print("Duplicados eliminados. Registros restantes:", len(df_live))

### Creamos una copia de trabajo

In [ ]:
# Copia del DataFrame limpio
df_live1 = df_live.copy()
df_live1

In [ ]:
df_live1.info()

In [ ]:
df_live1.describe()

### Corregimos tipos de datos y limpiamos espacios en columnas de texto

In [ ]:
# Eliminar espacios extra en columnas de texto
df_live1['Area']    = df_live1['Area'].str.strip()
df_live1['Item']    = df_live1['Item'].str.strip()
df_live1['Element'] = df_live1['Element'].str.strip()

# Convertir Year a datetime
df_live1['Year'] = pd.to_datetime(df_live1['Year'], format='%Y')

# Asegurar que Value sea numérico
df_live1['Value'] = pd.to_numeric(df_live1['Value'], errors='coerce')

df_live1.head()

### Obtenemos los valores únicos de la columna Element

In [ ]:
# Ver los indicadores únicos del dataset ganadero
valores_unicos_live = df_live1['Element'].unique()
print(valores_unicos_live)

### Validamos rangos lógicos — eliminamos valores negativos

In [ ]:
# Revisar rango de años
df_live1['Year_int'] = df_live1['Year'].dt.year
print("Rango de años:", df_live1['Year_int'].min(), "→", df_live1['Year_int'].max())

# Revisar valores negativos
print("Valores negativos en Value:", (df_live1['Value'] < 0).sum())

# Eliminar negativos si existen
df_live1 = df_live1[df_live1['Value'] >= 0]
print("Registros tras eliminar negativos:", len(df_live1))

### Eliminamos outliers con umbral en el percentil 99

In [ ]:
# Revisar estadísticas antes del filtro
df_live1.describe()

In [ ]:
# Aplicar umbral en percentil 99
umbral_live = df_live1['Value'].quantile(0.99)
df_live1 = df_live1[df_live1['Value'] < umbral_live]

print("Umbral aplicado:", umbral_live)
df_live1.describe()

### Renombramos columnas al español

In [ ]:
# Mapeo de indicadores al español
mapeo_element_live = {
    'Stocks':     'Existencias',
    'Production': 'Producción',
    'Yield':      'Rendimiento'
}

df_live1['Element'] = df_live1['Element'].map(mapeo_element_live).fillna(df_live1['Element'])

# Renombrar columnas base
df_live1 = df_live1.rename(columns={
    'Area':    'País',
    'Item':    'Animal',
    'Element': 'Indicador',
    'Unit':    'Unidad',
    'Value':   'Valor'
})

# Eliminar columna auxiliar Year_int
df_live1 = df_live1.drop(columns=['Year_int'])

df_live1.head()

### Separamos en dos DataFrames

In [ ]:
# DataFrame 1: columnas base de identificación
df_live1a = df_live1[['País', 'Animal', 'Indicador', 'Year', 'Unidad', 'Valor']].copy()

# DataFrame 2: tabla pivote por indicador
df_live1b = df_live1.pivot_table(
    index=['País', 'Animal', 'Year'],
    columns='Indicador',
    values='Valor',
    aggfunc='mean'
).reset_index()

print("df_live1a shape:", df_live1a.shape)
print("df_live1b shape:", df_live1b.shape)
df_live1b.head()

---
# 📊 Visualización Global — live1.csv

In [ ]:
# Valor medio por año
df_live1a['Year_int'] = df_live1a['Year'].dt.year
prod_live_año = df_live1a.groupby('Year_int')['Valor'].mean().reset_index()

plt.figure(figsize=(15, 6))
sns.lineplot(data=prod_live_año, x='Year_int', y='Valor')
plt.title('Valor Medio por Año — Ganadería (1961–2020)')
plt.xlabel('Año')
plt.ylabel('Valor Medio')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 países por valor medio
top_paises_live = df_live1a.groupby('País')['Valor'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_paises_live.values, y=top_paises_live.index)
plt.title('Top 10 Países por Valor Medio — Ganadería')
plt.xlabel('Valor Medio')
plt.ylabel('País')
plt.tight_layout()
plt.show()

In [ ]:
# Valor medio por indicador
por_indicador_live = df_live1a.groupby('Indicador')['Valor'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=por_indicador_live.values, y=por_indicador_live.index)
plt.title('Valor Medio por Indicador — Ganadería')
plt.xlabel('Valor Medio')
plt.ylabel('Indicador')
plt.tight_layout()
plt.show()

---
# ✅ Resumen Final del Proceso de Limpieza

| Paso | crop1.csv | live1.csv |
|------|-----------|----------|
| Carga de datos | ✅ | ✅ |
| Exploración inicial (head, info) | ✅ | ✅ |
| Conteo de nulos | ✅ 129.500 nulos en Value | ✅ 2.264 nulos en Value |
| Eliminación de nulos | ✅ dropna | ✅ dropna |
| Verificación post-limpieza | ✅ | ✅ |
| Eliminación de duplicados | ✅ | ✅ |
| Copia de trabajo | ✅ df_crop1 | ✅ df_live1 |
| Corrección de tipos (Year, Value) | ✅ | ✅ |
| Limpieza de espacios en texto | ✅ | ✅ |
| Eliminación de valores negativos | ✅ | ✅ |
| Eliminación de outliers (p99) | ✅ | ✅ |
| Renombrado de columnas al español | ✅ | ✅ |
| Separación en dos DataFrames | ✅ df_crop1a / df_crop1b | ✅ df_live1a / df_live1b |
| Visualizaciones | ✅ 3 gráficas | ✅ 3 gráficas |